In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog databricks_ecom_project")

In [0]:
spark.sql("create schema if not exists silver_schema")

In [0]:
silver_run_id = str(uuid.uuid4())
print("Current Silver Run Id:", silver_run_id)


## Step-2 - Silver control Table
this table stores Latest Silver processing state for each entity

it helps the pipeline remember:
- the latest Bronze run already processed by Silver
- the Latest Bronze ingestion timestamp already processed
- how many rows were mereged in the latest Silver run


In [0]:


spark.sql("""
CREATE TABLE databricks_ecom_project.silver_schema.processing_control (
    layer STRING,
    entity_name STRING,
    last_processed_bronze_run_id STRING,
    last_processed_bronze_ingested_at TIMESTAMP,
    rows_merged BIGINT,
    rows_status STRING,
    silver_run_id STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

spark.table("databricks_ecom_project.silver_schema.processing_control").printSchema()

## STEP4 - Helper Functions

This cell contains reusable functions:
- `upsert_to_silver` merges cleaned/transformed woes into silver target table
- `get_last_proccessed_ingested_at()` reads the Silver watermark
- `upsert_silver_control` updates to silver control table
- `get_incremental_bronze()` reads only new bronze rows that silver has not processed yet

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt=DeltaTable.forName(spark,target_table)
        (dt.alias("target")
         .merge(df_source.alias("source"),f"target.{join_key} = source.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)
         
         
         

In [0]:
def get_last_processed_ingested_at(entity_name:str):
    ctrl = (spark.table("databricks_ecom_project.silver_schema.processing_control")
           .filter(
               (F.col("layer") == "silver")&
               (F.col("entity_name") == entity_name)&
               (F.col("rows_status") == "SUCCESS")
           )
           .orderBy(F.col("updated_at").desc())
           .limit(1))
    rows = ctrl.collect()
    if not rows:
        return None
    
    return rows[0]["last_processed_bronze_ingested_at"]


In [0]:
from delta.tables import DeltaTable
from datetime import datetime

def upsert_silver_control(
    entity_name,
    last_processed_bronze_run_id,
    last_processed_bronze_ingested_at,
    rows_merged
):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.utcnow()
            )
        ],
        schema="""
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            rows_status string,
            silver_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(
        spark,
        "databricks_ecom_project.silver_schema.processing_control"
    )

    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"),
            "t.layer = s.layer AND t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(set={
            "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
            "rows_merged": "s.rows_merged",
            "rows_status": "s.rows_status",
            "silver_run_id": "s.silver_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsert(values={
            "layer": "s.layer",
            "entity_name": "s.entity_name",
            "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
            "rows_merged": "s.rows_merged",
            "rows_status": "s.rows_status",
            "silver_run_id": "s.silver_run_id",
            "updated_at": "s.updated_at"
        })
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at = get_last_processed_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df,last_ingested_at
    
    return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)),last_ingested_at

## Step4 - oders incremental processing

This cell process orders frm Bronze to silver.

it does the following:
- reads only new Bronze order rows
- cleans vaules like `order_status` and `order_amount`
-keeps only the latest version per `order_id`
- validated bussiness rules
- sends bad rows to quarantine
- merges good rows into `orders_transformed`

In [0]:
df_raw = spark.sql("select * from databricks_ecom_project.bronze_schema.orders_raw")
display(df_raw)

In [0]:
#step-4 Orders incremental processing
#Read only the bronze orders rows that silver has not processed 
orders_inc, last_orders_ingested_at = get_incremental_bronze(
    "databricks_ecom_project.bronze_schema.orders_raw", "orders"
)

#count the incremental order rows entering silver in the run
orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver = {orders_inc_count}")

#only run silver order cleaning and validation when there are new bronze order rows
if orders_inc_count > 0:
    #only run silver order cleaning and validation when there are new Bronze order rows
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    #start the silver order cleaning pipeline.This block standardizes and deduplicates raw order records
    orders_cleaned = (
        orders_inc
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status", F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status")))
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
        .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL", "?", ""), None).otherwise(F.col("order_amount")))
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        #Assign a row number inside each bussines key so we keep only latest version of the record
        .withColumn("row_rank", F.row_number().over(order_window))
        #keep only the latest for each bussines key
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    #Merge the cleaned and validated silver dataset into delta target table
    upsert_to_silver(
        orders_cleaned,
        "databricks_ecom_project.silver_schema.orders_cleaned",
        "order_id"
    )

    #Apply silver data quality rules to the cleaned order records
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            F.when(F.col("customer_id").isNull(), "verify_customer_id")
            .when(F.col("product_id").isNull(), "verify_product_id")
            .when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), "verify_order_amount")
            .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""), "verify_order_status")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), F.lit(True))
            .otherwise(F.lit(False))
        )
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_year", F.year("created_at"))
        .withColumn("order_month", F.month("created_at"))
        .withColumn("order_day", F.dayofmonth("created_at"))
        .withColumn("order_dow", F.date_format("created_at", "E"))
    )

    #keep only valid order rows for the transformed Silver table
    orders_good = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No Issues")

    #send invalid orders rows to the quarantine dataset for manual review
    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    #merge the cleaned and validated dataset into delta target table
    upsert_to_silver(
        orders_good,
        "databricks_ecom_project.silver_schema.orders_transformed",
        "order_id"
    )

    #Append the bad data rows to quarantite table insted of loosing them
    orders_bad.write.format("delta").mode("append").saveAsTable(
        "databricks_ecom_project.silver_schema.orders_quarantine"
    )

    mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

    mx_run = (
        orders_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    print("No New records Bronze rows for Silver")

    upsert_silver_control(
        "orders",
        None,
        last_orders_ingested_at,
        orders_inc_count
    )

In [0]:
%sql
select * from databricks_ecom_project.silver_schema.orders_cleaned;

In [0]:
%sql
select * from databricks_ecom_project.silver_schema.orders_quarantine;

## Step-5 Products incremental processing
This cell processes products from Bronze to Silver
 it handles
- Product name cleanup
- category standardization
- price cleanup and numeric conversion
- latest-record selection per `product_id`
- data quality validation
- quarantine for bad rows
- merge into silver current-state tables

In [0]:
%sql
select * from databricks_ecom_project.bronze_schema.products_raw;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

products_inc, last_products_ingested_at = get_incremental_bronze(
    "databricks_ecom_project.bronze_schema.products_raw",
    "products"
)

products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver = {products_inc_count}")

if products_inc_count > 0:

    product_window = Window.partitionBy("product_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    products_cleaned = (
        products_inc
        .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))
        .withColumn("product_name", F.regexp_replace(F.col("product_name"), r"[-_]", " "))
        .withColumn(
            "product_name",
            F.when(F.col("product_name") == "", F.lit(None)).otherwise(F.col("product_name"))
        )
        .withColumn(
            "category",
            F.when(
                F.upper(F.trim(F.col("category"))).contains("ELECTRNICS"),
                "ELECTRONICS"
            ).otherwise(F.upper(F.trim(F.col("category"))))
        )
        .withColumn("price", F.trim(F.col("price")))
        .withColumn("price", F.regexp_replace(F.col("price"), r"\$", ""))
        .withColumn("price", F.regexp_replace(F.col("price"), ",", "."))
        .withColumn("price", F.regexp_replace(F.col("price"), r"\s+", ""))
        .withColumn("price", F.expr("try_cast(price as double)"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        .withColumn("row_rank", F.row_number().over(product_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    products_validated = (
        products_cleaned
        .withColumn(
            "to_be_verified_by_products_team",
            F.when(F.col("product_name").isNull(), "verify_product_name")
             .when(F.col("category").isNull(), "verify_category")
             .when(F.col("price").isNull() | (F.col("price") <= 0), "verify_price")
             .otherwise("No Issues")
        )
        .withColumn(
            "check_product_price",
            F.when(
                F.col("price").isNull() | (F.col("price") <= 0),
                "Invalid_price"
            ).otherwise("valid_price")
        )
    )

    products_good = products_validated.filter(
        (F.col("to_be_verified_by_products_team") == "No Issues") &
        (F.col("check_product_price") == "valid_price")
    )

    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")

    products_bad = (
        products_validated
        .filter(
            (F.col("to_be_verified_by_products_team") != "No Issues") |
            (F.col("check_product_price") == "Invalid_price")
        )
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    upsert_to_silver(
        products_good,
        "databricks_ecom_project.silver_schema.products_transformed",
        "product_id"
    )

    products_bad.write.format("delta").mode("append").saveAsTable(
        "databricks_ecom_project.silver_schema.products_quarantine"
    )

    mx_ingested = products_inc.agg(
        F.max("bronze_ingested_at").alias("mx")
    ).collect()[0]["mx"]

    mx_run = (
        products_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control(
        "products",
        mx_run,
        mx_ingested,
        products_good.count()
    )

else:
    print("No new products Bronze rows to silver.")
    upsert_silver_control(
        "products",
        None,
        last_products_ingested_at,
        products_inc_count
    )

In [0]:
%sql 
select * from databricks_ecom_project.silver_schema.products_cleaned;

In [0]:
%sql 
select * from databricks_ecom_project.silver_schema.products_quarantine;

## Step-6 Payments incremental processing
This cell process paymentsfrom Bronze to Silver

It Cleans :
- `payments_status`
- `paid_amount`
- `processed_at`

Then it validates records,quarantites bad rows, and merge valid rows into silver transformed payments table

In [0]:
%sql
select * from databricks_ecom_project.bronze_schema.payments_raw;

In [0]:


payments_inc,last_payments_ingested_at = get_incremental_bronze("databricks_ecom_project.bronze_schema.payments_raw","payments")

payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver = {payments_inc_count}")

if payments_inc_count >0:

    payment_window = Window.partitionBy("payment_id").orderBy(F.col("processed_at").cast("timestamp").desc(),
                                                               F.col("bronze_ingested_at").desc())

    payments_cleaned = (
        payments_inc
        .withColumn("payment_status",F.upper(F.trim(F.col("payment_status"))))
        .withColumn("payment_status",F.when(F.col("payment_status") == "",F.lit(None)).otherwise(F.col("payment_status")))
        .withColumn("paid_amount",F.trim(F.col("paid_amount")))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"), r"\$",""))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),",","."))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"), r"\s+",""))
        .withColumn("paid_amount",F.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at",F.to_timestamp("processed_at"))
        .withColumn("row_rank",F.row_number().over(payment_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id",F.lit(silver_run_id))
    )

    upsert_to_silver(
        payments_cleaned,
        "databricks_ecom_project.silver_schema.payments_cleaned","payment_id")
    
    payments_validated =(
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payment_team",
            F.when(F.col("payment_status").isNull(), "verify_payment_status")
             .when(F.col("order_id").isNull(),"verify_orderr_id")
             .when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), "verify_paid_amount")
             .otherwise("No Issues")
        )
        .withColumn("check_paid_amount",
                    F.when(F.col("paid_amount").isNull() | (F.col("paid_amount") <=0), F.lit(True)).otherwise(F.lit(False)))
    )
    payments_good = payments_validated.filter(F.col("to_be_verified_by_payment_team") == "No Issues")

    payments_bad =(
        payments_validated.filter(
            F.col("to_be_verified_by_payment_team") != "No Issues").withColumn("quarantine_ts",F.current_timestamp())
        )
    upsert_to_silver(
        payments_good,
        "databricks_ecom_project.silver_schema.payments_transformed",
        "payment_id"
    )

    payments_bad.write.format("delta").mode("append").saveAsTable(
        "databricks_ecom_project.silver_schema.payments_quarantine"
    )

    mx_ingested = payments_inc.agg(
        F.max("bronze_ingested_at").alias("mx")
    ).collect()[0]["mx"]

    mx_run = (
        payments_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control(
        "payments",
        mx_run,
        mx_ingested,
        payments_good.count()
    )

else:
    print("No new payments Bronze rows to silver.")
    upsert_silver_control(
        "payments",
        None,
        last_payments_ingested_at,
        payments_inc_count
    )

       
    
        



                    
        

In [0]:
%sql 
select * from databricks_ecom_project.silver_schema.payments_cleaned;

In [0]:
%sql 
select * from databricks_ecom_project.silver_schema.payments_quarantine;

In [0]:
print("Products transformed count:", spark.sql("select count(*) from databricks_ecom_project.silver_schema.products_transformed").collect()[0][0])
print("Orders transformed count:", spark.sql("select count(*) from databricks_ecom_project.silver_schema.orders_transformed").collect()[0][0])
print("Payments transformed count:", spark.sql("select count(*) from databricks_ecom_project.silver_schema.payments_transformed").collect()[0][0])
display(spark.sql("SELECT * FROM databricks_ecom_project.silver_schema.processing_control").orderBy("entity_name"))